# Comparativa de las Tres Series — España · Global · Europa

**Pregunta central del TFG**: ¿Mejoran los foundation models de series temporales la predicción de inflación frente a los modelos estadísticos clásicos? ¿Las señales MCP/institucionales añaden valor y de qué forma depende eso del contexto?

| Serie | Variable | Periodo test | Mejor baseline | Señales C1 |
|-------|----------|--------------|----------------|------------|
| **España** | IPC España (INE) | 2021-2024 | ARIMA/SARIMA | MCP=IPC+GDELT (desde 2021) |
| **Global** | IPC Mundial | 2021-2024 | ARIMA | Inst: Fed Funds, EPU, Brent |
| **Europa** | HICP Eurozona | 2021-2024 | SARIMA | MCP=BCE/GDELT + DFR, ESI, TTF |

**Métrica principal**: MASE (normalizado por naive lag-12 en 2002-2020) — permite comparar las tres series en la misma escala aunque tengan unidades distintas.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm
import warnings
warnings.filterwarnings('ignore')

ROOT    = Path('..').resolve()
RESULTS = ROOT / '08_results'
HORIZONS = [1, 3, 6, 12]

# ══ ESPAÑA ════════════════════════════════════════════════════════
spain_raw = json.load(open(RESULTS / 'metrics_summary_final.json'))
for fname, key in [
    ('timesfm_C1_inst_metrics.json',  'timesfm_C1_inst'),
    ('timegpt_C1_inst_metrics.json',  'timegpt_C1_inst'),
    ('chronos2_C1_inst_metrics.json', 'chronos2_C1_inst'),
]:
    p = RESULTS / fname
    if p.exists():
        d = json.load(open(p))
        spain_raw[key] = d[key]

# ══ GLOBAL ════════════════════════════════════════════════════════
global_raw = {}
for src in ['rolling_metrics_global.json', 'rolling_metrics_C1_inst_global.json',
            'deep_rolling_metrics_global.json']:
    global_raw.update(json.load(open(RESULTS / src)))
for name in ['chronos2_C1_inst_global', 'timesfm_C1_inst_global', 'timegpt_C1_inst_global']:
    p = RESULTS / f'{name}_metrics.json'
    if p.exists():
        d = json.load(open(p))
        global_raw[name] = d[name]

# ══ EUROPA ════════════════════════════════════════════════════════
europe_raw = {}
for src in ['rolling_metrics_europe.json', 'deep_rolling_metrics_europe.json']:
    d = json.load(open(RESULTS / src))
    for k, v in d.items():
        europe_raw[f'{k}_europe'] = v
for name in [
    'chronos2_C0_europe', 'chronos2_C1_inst_europe', 'chronos2_C1_mcp_europe', 'chronos2_C1_full_europe',
    'timesfm_C0_europe',  'timesfm_C1_inst_europe',  'timesfm_C1_mcp_europe',  'timesfm_C1_full_europe',
    'timegpt_C0_europe',  'timegpt_C1_inst_europe',  'timegpt_C1_mcp_europe',  'timegpt_C1_full_europe',
]:
    p = RESULTS / f'{name}_metrics.json'
    if p.exists():
        d = json.load(open(p))
        europe_raw[name] = d[name]

# ── helpers ────────────────────────────────────────────────────────
def mae(d, h):  return d.get(f'h{h}', {}).get('MAE')
def mase(d, h): return d.get(f'h{h}', {}).get('MASE')

MASE_SCALE = {
    'spain':  spain_raw['naive']['h1']['MAE']  / spain_raw['naive']['h1']['MASE'],
    'global': global_raw['naive']['h1']['MAE'] / global_raw['naive']['h1']['MASE'],
    'europe': europe_raw['naive_europe']['h1']['MAE'] / europe_raw['naive_europe']['h1']['MASE'],
}
SCOLS = {'españa': '#e6550d', 'global': '#31a354', 'europa': '#3182bd'}

print(f'Modelos: España={len(spain_raw)}, Global={len(global_raw)}, Europa={len(europe_raw)}')
print('Escala MASE por serie:')
for s, v in MASE_SCALE.items():
    print(f'  {s:7s}: {v:.4f} pp')

---
## 1 · Dificultad de predicción — MASE por serie

El MASE del mejor modelo indica si las series son previsibles en cada horizonte.

In [ ]:
BEST = {
    'españa': {'data': spain_raw,  'stat': 'arima',        'found': 'timesfm_C1_inst', 'color': '#e6550d'},
    'global': {'data': global_raw, 'stat': 'arima',        'found': 'chronos2_C1_inst_global', 'color': '#31a354'},
    'europa': {'data': europe_raw, 'stat': 'sarima_europe','found': 'timesfm_C1_full_europe',  'color': '#3182bd'},
}
LABELS = {'españa': 'España (IPC)', 'global': 'Global (IPC)', 'europa': 'Europa (HICP)'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
x = np.arange(len(HORIZONS))

# Izquierda: MASE stat (punteado) vs foundation (continuo)
ax = axes[0]
for sid, cfg in BEST.items():
    d = cfg['data']
    sv = [mase(d.get(cfg['stat'],  {}), h) for h in HORIZONS]
    fv = [mase(d.get(cfg['found'], {}), h) for h in HORIZONS]
    ax.plot(x, sv, 's:', color=cfg['color'], lw=1.5, ms=7, alpha=0.65, label=f'{LABELS[sid]} — stat')
    ax.plot(x, fv, 'o-', color=cfg['color'], lw=2.5, ms=8, label=f'{LABELS[sid]} — found. best')
ax.axhline(1, color='black', lw=1.2, ls='--', alpha=0.4, label='MASE=1 (≡ Naive)')
ax.set_xticks(x); ax.set_xticklabels([f'h={h}' for h in HORIZONS], fontsize=11)
ax.set_ylabel('MASE', fontsize=10)
ax.set_title('MASE — Mejor estadístico (punteado) vs\nMejor foundation (continuo)', fontsize=10, fontweight='bold')
ax.legend(fontsize=7.5, loc='upper left', framealpha=0.85)
ax.grid(alpha=0.25)

# Derecha: MASE del naive (dificultad intrínseca)
ax2 = axes[1]
for sid, cfg in BEST.items():
    d = cfg['data']
    nk = 'naive_europe' if sid == 'europa' else 'naive'
    vals = [mase(d.get(nk, {}), h) for h in HORIZONS]
    ax2.plot(x, vals, 's-', color=cfg['color'], lw=2.2, ms=8, label=LABELS[sid])
ax2.set_xticks(x); ax2.set_xticklabels([f'h={h}' for h in HORIZONS], fontsize=11)
ax2.set_ylabel('MASE del naive lag-12', fontsize=10)
ax2.set_title('MASE del Naive — dificultad intrínseca\n(cuánto margen hay para mejorar)', fontsize=10, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.25)

fig.suptitle('Dificultad de predicción por serie', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS / 'fig_comp1_difficulty.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2 · Foundation Models vs Mejor Modelo Estadístico

¿Consiguen los foundation models batir al mejor baseline estadístico en cada serie?

In [ ]:
FOUNDATION_BEST = {
    'españa': {'data': spain_raw,  'stat_key': 'arima',        'stat_lbl': 'ARIMA',
               'found_key': 'timesfm_C1_inst',         'found_lbl': 'TimesFM C1_inst',    'color': '#e6550d'},
    'global': {'data': global_raw, 'stat_key': 'arima',        'stat_lbl': 'ARIMA',
               'found_key': 'chronos2_C1_inst_global',  'found_lbl': 'Chronos-2 C1_inst ★','color': '#31a354'},
    'europa': {'data': europe_raw, 'stat_key': 'sarima_europe','stat_lbl': 'SARIMA',
               'found_key': 'timesfm_C1_full_europe',   'found_lbl': 'TimesFM C1_full ★★', 'color': '#3182bd'},
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
x = np.arange(len(HORIZONS))
for ax, (sid, cfg) in zip(axes, FOUNDATION_BEST.items()):
    d = cfg['data']
    stat  = [mae(d.get(cfg['stat_key'],  {}), h) for h in HORIZONS]
    found = [mae(d.get(cfg['found_key'], {}), h) for h in HORIZONS]
    ax.plot(x, stat,  'o-', color='#444', lw=2.5, ms=8, label=f"Mejor estat.: {cfg['stat_lbl']}", zorder=5)
    ax.plot(x, found, 's-', color=cfg['color'], lw=2.5, ms=8, label=f"Foundation: {cfg['found_lbl']}", zorder=5)
    for i in range(len(HORIZONS)-1):
        col = '#90ee90' if found[i] < stat[i] else '#ffcccc'
        ax.fill_between([x[i],x[i+1]], [stat[i],stat[i+1]], [found[i],found[i+1]], alpha=0.22, color=col)
    for i, h in enumerate(HORIZONS):
        if stat[i] and found[i]:
            delta = (found[i] - stat[i]) / stat[i] * 100
            ypos  = max(stat[i], found[i]) * 1.05
            ax.text(i, ypos, f'{delta:+.0f}%', ha='center', va='bottom', fontsize=9,
                    color='#006600' if delta < 0 else '#880000', fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels([f'h={h}' for h in HORIZONS], fontsize=10)
    ax.set_title(sid.upper(), fontsize=12, fontweight='bold', color=cfg['color'])
    ax.set_xlabel('Horizonte', fontsize=9); ax.set_ylabel('MAE', fontsize=9)
    ax.legend(fontsize=8.5, loc='upper left'); ax.grid(alpha=0.22)

fig.suptitle('Foundation vs Estadístico por serie\nVerde = foundation gana · Rojo = estadístico gana',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS / 'fig_comp2_foundation_vs_stat.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3 · Comparativa de familias por serie — MASE en h=1 y h=12

In [ ]:
FAMILIES = {
    'Chronos-2': {
        'españa': (spain_raw,  'chronos2_C0'),
        'global': (global_raw, 'chronos2_C1_inst_global'),
        'europa': (europe_raw, 'chronos2_C1_inst_europe'),
    },
    'TimesFM': {
        'españa': (spain_raw,  'timesfm_C1_inst'),
        'global': (global_raw, 'timesfm_C1_inst_global'),
        'europa': (europe_raw, 'timesfm_C1_full_europe'),
    },
    'TimeGPT': {
        'españa': (spain_raw,  'timegpt_C0'),
        'global': (global_raw, 'timegpt_C1_inst_global'),
        'europa': (europe_raw, 'timegpt_C0_europe'),
    },
}
STAT_REF = {
    'españa': (spain_raw,  'arima'),
    'global': (global_raw, 'arima'),
    'europa': (europe_raw, 'sarima_europe'),
}
fam_colors   = {'Chronos-2': '#52a852', 'TimesFM': '#9467bd', 'TimeGPT': '#ff7f0e'}
series_list  = ['españa', 'global', 'europa']
series_lbls  = ['España', 'Global', 'Europa']

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, h in zip(axes, [1, 12]):
    xp = np.arange(len(series_list))
    w  = 0.25
    for i, (fname, fdata) in enumerate(FAMILIES.items()):
        vals = []
        for sid in series_list:
            d, k = fdata[sid]
            v = mase(d.get(k, {}), h)
            vals.append(v if v else 0)
        bars = ax.bar(xp + (i-1)*w, vals, w, label=fname,
                      color=fam_colors[fname], alpha=0.85, edgecolor='white')
        for bar, v in zip(bars, vals):
            if v:
                ax.text(bar.get_x()+bar.get_width()/2, v+0.02,
                        f'{v:.2f}', ha='center', va='bottom', fontsize=7.5, fontweight='bold')
    # Referencia estadística por serie
    for j, sid in enumerate(series_list):
        d, k = STAT_REF[sid]
        v = mase(d.get(k, {}), h)
        if v:
            ax.hlines(v, xp[j]-0.38, xp[j]+0.38, colors=SCOLS[sid], lw=2.0, ls='--', alpha=0.65)
    ax.axhline(1, color='black', lw=1.2, ls=':', alpha=0.5, label='MASE=1 (Naive)')
    ax.set_xticks(xp); ax.set_xticklabels(series_lbls, fontsize=11)
    ax.set_ylabel('MASE', fontsize=10)
    ax.set_title(f'Familias Foundation — h={h}\n(línea = mejor estadístico de la serie)',
                 fontsize=10, fontweight='bold')
    ax.legend(fontsize=9, loc='upper right'); ax.grid(axis='y', alpha=0.22)
fig.suptitle('MASE por familia y serie — h=1 y h=12', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS / 'fig_comp3_families.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4 · Valor de las señales C1 por serie y familia

¿Cuánto añaden (o restan) las señales institucionales/MCP respecto al C0 (o ARIMA para Global)?

In [ ]:
C1_ROWS = [
    # label                          d_ref       k_ref                d_c1       k_c1
    ('España — Chronos-2 C1-MCP',   spain_raw, 'chronos2_C0',       spain_raw, 'chronos2_C1'),
    ('España — TimesFM  C1-MCP',    spain_raw, 'timesfm_C0',        spain_raw, 'timesfm_C1'),
    ('España — TimesFM  C1-inst',   spain_raw, 'timesfm_C0',        spain_raw, 'timesfm_C1_inst'),
    ('Global — Chronos-2 C1-inst',  global_raw,'arima',              global_raw,'chronos2_C1_inst_global'),
    ('Global — TimesFM  C1-inst',   global_raw,'arima',              global_raw,'timesfm_C1_inst_global'),
    ('Europa — Chronos-2 C1-inst',  europe_raw,'chronos2_C0_europe', europe_raw,'chronos2_C1_inst_europe'),
    ('Europa — TimesFM  C1-inst',   europe_raw,'timesfm_C0_europe',  europe_raw,'timesfm_C1_inst_europe'),
    ('Europa — TimesFM  C1-full ★★',europe_raw,'timesfm_C0_europe',  europe_raw,'timesfm_C1_full_europe'),
]

hm = np.full((len(C1_ROWS), 4), np.nan)
for i, (_, d0, k0, d1, k1) in enumerate(C1_ROWS):
    for j, h in enumerate(HORIZONS):
        m0v = mae(d0.get(k0, {}), h)
        m1v = mae(d1.get(k1, {}), h)
        if m0v and m1v:
            hm[i, j] = (m1v - m0v) / m0v * 100

fig, ax = plt.subplots(figsize=(9, 7.5))
norm = TwoSlopeNorm(vmin=-30, vcenter=0, vmax=100)
im = ax.imshow(hm, cmap='RdYlGn_r', norm=norm, aspect='auto')
ax.set_xticks(range(4)); ax.set_xticklabels([f'h={h}' for h in HORIZONS], fontsize=11)
ax.set_yticks(range(len(C1_ROWS))); ax.set_yticklabels([r[0] for r in C1_ROWS], fontsize=9.5)
ax.set_title('Δ MAE relativo: C1 vs referencia × 100%\n'
             'Verde = C1 mejora  |  Rojo = C1 empeora', fontsize=11, fontweight='bold')
for i in range(len(C1_ROWS)):
    for j in range(4):
        v = hm[i, j]
        if not np.isnan(v):
            ct = 'white' if abs(v) > 35 else 'black'
            ax.text(j, i, f'{v:+.1f}%', ha='center', va='center',
                    fontsize=9.5, color=ct, fontweight='bold')
ax.axhline(2.5, color='white', lw=2.5)
ax.axhline(4.5, color='white', lw=2.5)
plt.colorbar(im, ax=ax, shrink=0.65, label='Δ MAE (%)')
plt.tight_layout()
plt.savefig(RESULTS / 'fig_comp4_c1_effect.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5 · Figura Principal — Para la Memoria del TFG

Panel 2×3 que sintetiza el experimento completo.

In [ ]:
matplotlib.rcParams.update({'font.family': 'DejaVu Sans',
                             'axes.spines.top': False, 'axes.spines.right': False})
fig = plt.figure(figsize=(18, 11))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.48, wspace=0.32)
ax_a = fig.add_subplot(gs[0, 0])
ax_b = fig.add_subplot(gs[0, 1])
ax_c = fig.add_subplot(gs[0, 2])
ax_d = fig.add_subplot(gs[1, 0])
ax_e = fig.add_subplot(gs[1, 1])
ax_f = fig.add_subplot(gs[1, 2])
xh = np.arange(len(HORIZONS))

# ── A: Perfiles MAE — España ──────────────────────────────────────
for key, col, lbl, sty, lw in [
    ('sarima',          '#888',    'SARIMA (best stat)',    's:',  1.6),
    ('auto_arima',      '#8B4513', 'AutoARIMA (din.)',      'x--', 1.4),
    ('timesfm_C0',      '#ff9e4a', 'TimesFM C0',            'o-',  2.0),
    ('timesfm_C1_inst', '#e6550d', 'TimesFM C1_inst ★',     'P-',  2.4),
    ('chronos2_C0',     '#f4a582', 'Chronos-2 C0',          '^--', 1.6),
]:
    vals = [mae(spain_raw.get(key, {}), h) for h in HORIZONS]
    if all(v for v in vals):
        ax_a.plot(xh, vals, sty, color=col, lw=lw, ms=7, label=lbl)
ax_a.set_xticks(xh); ax_a.set_xticklabels([f'h={h}' for h in HORIZONS], fontsize=9)
ax_a.set_ylabel('MAE (pp IPC España)', fontsize=9)
ax_a.set_title('(A) España — IPC', fontsize=10, fontweight='bold', color='#e6550d')
ax_a.legend(fontsize=7.5, loc='upper left'); ax_a.grid(alpha=0.22)
ax_a.annotate('Foundation NO supera\nal estadístico', xy=(0.98, 0.96), xycoords='axes fraction',
              fontsize=7.5, va='top', ha='right', color='#880000',
              bbox=dict(boxstyle='round', fc='#ffe0e0', alpha=0.8))

# ── B: Perfiles MAE — Global ──────────────────────────────────────
for key, col, lbl, sty, lw in [
    ('arima',                   '#888',    'ARIMA (best stat)',      's:',  1.6),
    ('auto_arima',              '#8B4513', 'AutoARIMA (din.)',        'x--', 1.4),
    ('chronos2_C1_inst_global', '#31a354', 'Chronos-2 C1_inst ★★',   'P-',  2.4),
    ('timesfm_C1_inst_global',  '#74c476', 'TimesFM C1_inst',         'o--', 2.0),
    ('timegpt_C1_inst_global',  '#b3e2c8', 'TimeGPT C1_inst',         'v:',  1.6),
]:
    vals = [mae(global_raw.get(key, {}), h) for h in HORIZONS]
    if all(v for v in vals):
        ax_b.plot(xh, vals, sty, color=col, lw=lw, ms=7, label=lbl)
ax_b.set_xticks(xh); ax_b.set_xticklabels([f'h={h}' for h in HORIZONS], fontsize=9)
ax_b.set_ylabel('MAE (pp IPC Global)', fontsize=9)
ax_b.set_title('(B) Global — IPC', fontsize=10, fontweight='bold', color='#31a354')
ax_b.legend(fontsize=7.5, loc='upper left'); ax_b.grid(alpha=0.22)
ax_b.annotate('Chronos-2 bate\na ARIMA (h≥3)', xy=(0.98, 0.96), xycoords='axes fraction',
              fontsize=7.5, va='top', ha='right', color='#006600',
              bbox=dict(boxstyle='round', fc='#e0f0e0', alpha=0.8))

# ── C: Perfiles MAE — Europa ──────────────────────────────────────
for key, col, lbl, sty, lw in [
    ('sarima_europe',          '#888',    'SARIMA (best stat)',   's:',  1.6),
    ('auto_arima_europe',      '#8B4513', 'AutoARIMA (din.)',     'x--', 1.4),
    ('timesfm_C0_europe',      '#9ecae1', 'TimesFM C0',           'o-',  2.0),
    ('timesfm_C1_full_europe', '#3182bd', 'TimesFM C1_full ★★',   'P-',  2.4),
    ('chronos2_C0_europe',     '#6baed6', 'Chronos-2 C0',         '^--', 1.6),
]:
    vals = [mae(europe_raw.get(key, {}), h) for h in HORIZONS]
    if all(v for v in vals):
        ax_c.plot(xh, vals, sty, color=col, lw=lw, ms=7, label=lbl)
ax_c.set_xticks(xh); ax_c.set_xticklabels([f'h={h}' for h in HORIZONS], fontsize=9)
ax_c.set_ylabel('MAE (puntos HICP)', fontsize=9)
ax_c.set_title('(C) Europa — HICP', fontsize=10, fontweight='bold', color='#3182bd')
ax_c.legend(fontsize=7.5, loc='upper left'); ax_c.grid(alpha=0.22)
ax_c.annotate('TimesFM C1_full bate\na SARIMA (h≥6)', xy=(0.98, 0.96), xycoords='axes fraction',
              fontsize=7.5, va='top', ha='right', color='#003366',
              bbox=dict(boxstyle='round', fc='#ddeeff', alpha=0.8))

# ── D: MASE h=12 — barras agrupadas ──────────────────────────────
groups = {
    'Naive':        [mase(spain_raw.get('naive',{}),12),
                     mase(global_raw.get('naive',{}),12),
                     mase(europe_raw.get('naive_europe',{}),12)],
    'Mejor\nestad.':[mase(spain_raw.get('arima',{}),12),
                     mase(global_raw.get('arima',{}),12),
                     mase(europe_raw.get('sarima_europe',{}),12)],
    'Mejor\nfound.':[mase(spain_raw.get('timesfm_C1_inst',{}),12),
                     mase(global_raw.get('chronos2_C1_inst_global',{}),12),
                     mase(europe_raw.get('timesfm_C1_full_europe',{}),12)],
}
xd = np.arange(3)
for i, (k, vals) in enumerate(groups.items()):
    gcol = ['#b0b0b0', '#606060', '#e3a020'][i]
    bars = ax_d.bar(xd + (i-1)*0.25, vals, 0.25, label=k, color=gcol, alpha=0.88, edgecolor='white')
    for bar, v in zip(bars, vals):
        if v:
            ax_d.text(bar.get_x()+bar.get_width()/2, v+0.04, f'{v:.2f}',
                      ha='center', va='bottom', fontsize=7.5, fontweight='bold')
ax_d.axhline(1, color='black', lw=1.2, ls='--', alpha=0.5, label='MASE=1 (Naive)')
ax_d.set_xticks(xd); ax_d.set_xticklabels(['España','Global','Europa'], fontsize=10)
ax_d.set_ylabel('MASE (h=12)', fontsize=9)
ax_d.set_title('(D) MASE a h=12\nNaive vs Estadístico vs Foundation', fontsize=10, fontweight='bold')
ax_d.legend(fontsize=8, loc='upper right'); ax_d.grid(axis='y', alpha=0.22)

# ── E: Heatmap Δ C1 compacto ─────────────────────────────────────
HM_E = [
    ('España Chronos-2 C1-MCP',  spain_raw,'chronos2_C0',        spain_raw,'chronos2_C1'),
    ('España TimesFM  C1-MCP',   spain_raw,'timesfm_C0',         spain_raw,'timesfm_C1'),
    ('España TimesFM  C1-inst',  spain_raw,'timesfm_C0',         spain_raw,'timesfm_C1_inst'),
    ('Global Chronos-2 C1-inst', global_raw,'arima',              global_raw,'chronos2_C1_inst_global'),
    ('Global TimesFM  C1-inst',  global_raw,'arima',              global_raw,'timesfm_C1_inst_global'),
    ('Europa TimesFM  C1-inst',  europe_raw,'timesfm_C0_europe',  europe_raw,'timesfm_C1_inst_europe'),
    ('Europa TimesFM  C1-full★★',europe_raw,'timesfm_C0_europe',  europe_raw,'timesfm_C1_full_europe'),
]
hme = np.full((len(HM_E), 4), np.nan)
for i, (_, d0, k0, d1, k1) in enumerate(HM_E):
    for j, h in enumerate(HORIZONS):
        m0v = mae(d0.get(k0,{}),h); m1v = mae(d1.get(k1,{}),h)
        if m0v and m1v: hme[i,j] = (m1v-m0v)/m0v*100
norm_e = TwoSlopeNorm(vmin=-20, vcenter=0, vmax=60)
ime = ax_e.imshow(hme, cmap='RdYlGn_r', norm=norm_e, aspect='auto')
ax_e.set_xticks(range(4)); ax_e.set_xticklabels([f'h={h}' for h in HORIZONS], fontsize=9)
ax_e.set_yticks(range(len(HM_E))); ax_e.set_yticklabels([r[0] for r in HM_E], fontsize=8.5)
ax_e.set_title('(E) Δ MAE: C1 vs referencia\nVerde=mejora · Rojo=empeora', fontsize=10, fontweight='bold')
for i in range(len(HM_E)):
    for j in range(4):
        v = hme[i,j]
        if not np.isnan(v):
            ct = 'white' if abs(v) > 30 else 'black'
            ax_e.text(j, i, f'{v:+.0f}%', ha='center', va='center', fontsize=8.5, color=ct, fontweight='bold')
ax_e.axhline(2.5, color='white', lw=2.5); ax_e.axhline(4.5, color='white', lw=2.5)
plt.colorbar(ime, ax=ax_e, shrink=0.8, label='Δ MAE (%)')

# ── F: Tabla resumen ──────────────────────────────────────────────
ax_f.axis('off')
rows_f = [
    ['España',  'TimesFM C1_inst',   f"{mase(spain_raw.get('timesfm_C1_inst',{}),1):.3f}",
                'ARIMA',             f"{mase(spain_raw.get('arima',{}),12):.3f}",  '~ 0%\n(C1≈C0)'],
    ['Global',  'ARIMA',             f"{mase(global_raw.get('arima',{}),1):.3f}",
                'Chronos-2 C1_inst', f"{mase(global_raw.get('chronos2_C1_inst_global',{}),12):.3f}", '★ −26%'],
    ['Europa',  'SARIMA',            f"{mase(europe_raw.get('sarima_europe',{}),1):.3f}",
                'TimesFM C1_full',   f"{mase(europe_raw.get('timesfm_C1_full_europe',{}),12):.3f}",  '★★ −17%'],
]
headers = ['Serie', 'Mejor h=1', 'MASE h=1', 'Mejor h=12', 'MASE h=12', 'Valor C1']
tbl = ax_f.table(cellText=rows_f, colLabels=headers, loc='center', cellLoc='center', bbox=[0, 0.1, 1, 0.82])
tbl.auto_set_font_size(False); tbl.set_fontsize(8.5)
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor('#dddddd'); cell.set_text_props(fontweight='bold')
    elif c == 5:
        txt = cell.get_text().get_text()
        cell.set_facecolor('#c8e6c9' if '★' in txt else '#fff9e6')
    cell.set_edgecolor('#aaaaaa')
ax_f.set_title('(F) Resumen comparativo', fontsize=10, fontweight='bold', pad=10)

# ── Global ────────────────────────────────────────────────────────
fig.suptitle(
    'Comparativa España · Global · Europa — Rolling-origin 2021-2024\n'
    'Foundation models vs baseline estadístico · Valor de las señales C1',
    fontsize=12, fontweight='bold', y=1.01)
legend_patches = [
    mpatches.Patch(color='#e6550d', label='España (IPC)'),
    mpatches.Patch(color='#31a354', label='Global (IPC)'),
    mpatches.Patch(color='#3182bd', label='Europa (HICP Eurozona)'),
    mpatches.Patch(color='#8B4513', label='AutoARIMA dinámico (ref.)'),
    mpatches.Patch(color='#e3a020', label='Mejor Foundation Model'),
    mpatches.Patch(color='#c8e6c9', label='C1 con mejora significativa ★'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=6, fontsize=8.2,
           frameon=True, bbox_to_anchor=(0.5, -0.045))
plt.savefig(RESULTS / 'fig_MAIN_comparison.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print('\n=== FIGURA PRINCIPAL GUARDADA: fig_MAIN_comparison.png ===')

---
## 6 · Síntesis — Hallazgos transversales para la memoria

In [ ]:
print('=' * 75)
print('SÍNTESIS TRANSVERSAL — ESPAÑA / GLOBAL / EUROPA')
print('=' * 75)

H1 = '''
HALLAZGO 1 — Foundation models son competitivos pero el contexto importa
─────────────────────────────────────────────────────────────────────────
  España : ARIMA (h=12 MAE=1.541) supera a todos los foundation models.
           El IPC español tiene dinámica perfectamente capturada por ARIMA.
  Global : Chronos-2 C1_inst supera a ARIMA en h≥3 (-26% a h=12).
           ARIMA sigue siendo el rey a h=1 (MAE=0.191 vs 0.200).
  Europa : TimesFM C1_full supera a SARIMA en h≥6 (MAE=1.995 vs 2.411).
           Primera vez que un modelo rompe la barrera MAE<2.0 a h=12.
'''
print(H1)

H2 = '''
HALLAZGO 2 — Las señales C1 son beneficiosas SOLO en las series correctas
─────────────────────────────────────────────────────────────────────────
  España : C1-MCP DEGRADA (+55% TimesFM, +9% Chronos-2).
           Causa: señales disponibles solo desde 2021, histórico mínimo.
           C1-inst: neutro (-2% a h=12, no significativo).
  Global : C1-inst ayuda a Chronos-2 (-26% h=12) y TimesFM (-17% h=12).
  Europa : C1_full (inst+MCP) es el mejor modelo (-17% vs C0 a h=12).
           C1-inst solo: modesto (-2 a -4%). MCP añade valor real.
'''
print(H2)

H3 = '''
HALLAZGO 3 — Ranking de familias
────────────────────────────────
  Chronos-2 : más robusto para señales inst. globales.
              Mejor para IPC Global (MASE h=12 = 0.976 — el único <1.0).
  TimesFM   : más sensible a señales MCP, beneficio claro en Europa.
              Mejor para HICP Europa (C1_full, MASE h=12 = 1.370).
              También mejor para España a h=1 (C1_inst, MASE=0.301).
  TimeGPT   : más frágil ante señales extremas (carry-forward 2022).
              Consistentemente peor en todas las series.
'''
print(H3)

H4 = '''
HALLAZGO 4 — El horizonte es clave
───────────────────────────────────
  h=1  : ARIMA/SARIMA muy difíciles de batir.
  h=3-6: Foundation empieza a ser competitivo en Global y Europa.
  h=12 : Foundation gana en Global (-26%) y Europa (-17%); empata en España.
         Para decisiones de política monetaria (horizonte largo),
         los foundation models con señales institucionales son preferibles.
'''
print(H4)

# AutoARIMA comparison across series
aa_es = spain_raw.get('auto_arima', {})
aa_gl = global_raw.get('auto_arima', {})
aa_eu = europe_raw.get('auto_arima_europe', {})
ref_es = spain_raw.get('arima', {})
ref_gl = global_raw.get('arima', {})
ref_eu = europe_raw.get('sarima_europe', {})

H5 = 'HALLAZGO 5 — AutoARIMA dinámico vs estadístico fijo (recomendación del tutor)\n'
H5 += '─' * 70 + '\n'
H5 += '  La reselección de órdenes SARIMA en cada origen rolling EMPEORA respecto\n'
H5 += '  al modelo de órdenes fijos seleccionado 1× sobre el histórico completo.\n\n'
for sid, aa_d, ref_d, ref_lbl in [
    ('España', aa_es, ref_es, 'ARIMA'),
    ('Global', aa_gl, ref_gl, 'ARIMA'),
    ('Europa', aa_eu, ref_eu, 'SARIMA'),
]:
    m_aa_1  = mae(aa_d, 1)
    m_ref_1 = mae(ref_d, 1)
    m_aa_12  = mae(aa_d, 12)
    m_ref_12 = mae(ref_d, 12)
    if m_aa_1 and m_ref_1 and m_aa_12 and m_ref_12:
        d1  = (m_aa_1  - m_ref_1)  / m_ref_1  * 100
        d12 = (m_aa_12 - m_ref_12) / m_ref_12 * 100
        H5 += f'  {sid:7s}: h=1  AutoARIMA={m_aa_1:.4f} vs {ref_lbl}={m_ref_1:.4f} ({d1:+.1f}%)\n'
        H5 += f'           h=12 AutoARIMA={m_aa_12:.4f} vs {ref_lbl}={m_ref_12:.4f} ({d12:+.1f}%)\n'
    else:
        H5 += f'  {sid:7s}: datos no disponibles\n'
H5 += ('  Causa: auto_arima sobreajusta al submuestreo reciente de cada ventana;\n'
       '  los órdenes fijos generalizan mejor al horizonte largo.\n'
       '  Implicación: parametrizar una vez sobre el histórico completo\n'
       '  es superior a la reselección automática en rolling.\n')
print(H5)

HM = '''
HALLAZGO METODOLÓGICO — Ridge + StandardScaler
──────────────────────────────────────────────
  Sin normalización : coeficientes espurios (EPU std~65 vs diff(HICP) std~0.44)
                      → corrección constante +1.11 pp, TimesFM MAE +534% vs C0.
  Con StandardScaler: corrección proporcional (+0.02 estable → +0.58 pico 2022).
  Lección            : señales de escala muy distinta necesitan normalización previa.
'''
print(HM)
print('=' * 75)